# Inpactor3-SegNT · Pipeline COMPLETO en Colab

**Diferencia con el piloto**: usa TAIR10 completo (5 cromosomas, ~119 Mb) y
las anotaciones reales de Inpactor2 sobre el genoma entero (~109 elementos).

**Split del informe**:
- Entrena: Chr1, Chr2, Chr3
- Valida: Chr4
- Examen: Chr5

**Etiquetas**: 14 clases (fondo + 13 linajes InpactorDB).

**Métricas nuevas**:
- F1 por ventana (como el piloto)
- IoU medio de coordenadas
- Matriz de confusión de linaje

**Antes de correr**: `Runtime → Change runtime type → T4 GPU`.

Tiempo total estimado: **~1 hora** en GPU T4.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Clonar repo e instalar

In [ ]:
%cd /content
!rm -rf Inpactor3-SegNT   # limpia por si existe de una sesión anterior
!git clone https://github.com/Inpactor3/Inpactor3-SegNT.git
%cd Inpactor3-SegNT
!pip install -q -e .

## 3. Bajar TAIR10 completo (5 cromosomas, ~5 min)

In [ ]:
!mkdir -p data/tair10_full && cd data/tair10_full && \
for CHR in 1 2 3 4 5; do \
  wget -q --show-progress \
  "https://ftp.ensemblgenomes.org/pub/plants/release-58/fasta/arabidopsis_thaliana/dna/Arabidopsis_thaliana.TAIR10.dna_sm.chromosome.${CHR}.fa.gz"; \
done && \
gunzip -f *.gz && \
> TAIR10.fasta && \
for CHR in 1 2 3 4 5; do \
  echo ">Chr${CHR}" >> TAIR10.fasta; \
  grep -v '^>' "Arabidopsis_thaliana.TAIR10.dna_sm.chromosome.${CHR}.fa" \
    | tr 'acgtn' 'ACGTN' \
    | sed 's/[^ACGTN]/N/g' >> TAIR10.fasta; \
done && \
grep '^>' TAIR10.fasta && \
echo -n 'Total bases: ' && grep -v '^>' TAIR10.fasta | tr -d '\n' | wc -c

## 4. Verificar ground truth Inpactor2

Este archivo (`Inpactor2_predictions.tab` sobre TAIR10 completo) debe estar
en el repo. Si no está aún, súbelo con la celda de `files.upload()` que
aparece abajo.

In [ ]:
import os
gt_path = 'data/tair10_full/inpactor2_out/Inpactor2_predictions.tab'
if os.path.exists(gt_path):
    !wc -l {gt_path}
    !echo '---primeras 5---'
    !head -5 {gt_path}
    !echo '---distribución por cromosoma---'
    !awk '{{print $1}}' {gt_path} | sort | uniq -c
else:
    print(f'FALTA: {gt_path}')
    print('Sube el archivo con la siguiente celda.')

In [ ]:
# SOLO si el archivo NO estaba en el repo: subir manualmente.
from google.colab import files
print('Sube Inpactor2_predictions.tab:')
uploaded = files.upload()
!mkdir -p data/tair10_full/inpactor2_out
!mv Inpactor2_predictions.tab data/tair10_full/inpactor2_out/
!ls -lh data/tair10_full/inpactor2_out/

## 5. Entrenar (15 épocas, ~30-45 min GPU T4)

In [ ]:
!python -m inpactor3_segnt.train --config configs/nt_50m_tair10_full.yaml

## 6. Inferencia sobre Chr5 (examen final, ~5 min)

In [ ]:
!python -m inpactor3_segnt.predict \
    --checkpoint models/best.pt \
    --genome data/tair10_full/TAIR10.fasta \
    --scaffold Chr5 \
    --out results/chr5_predictions.tab

## 7. Evaluar contra Inpactor2 (F1 + IoU + confusión de linaje)

In [ ]:
!python -m inpactor3_segnt.evaluate \
    --pred results/chr5_predictions.tab \
    --truth data/tair10_full/inpactor2_out/Inpactor2_predictions.tab \
    --scaffold Chr5 \
    --genome data/tair10_full/TAIR10.fasta \
    --multiclass \
    --report results/chr5_report.md

In [ ]:
!cat results/chr5_report.md

## 8. Descargar todo a tu PC

In [ ]:
from google.colab import files
files.download('models/best.pt')
files.download('results/chr5_report.md')
files.download('results/chr5_predictions.tab')
files.download('logs/train_history.json')